# The payoff — answering the research question with tidy data

**Research question:** for this coupling reaction, which combination of catalyst and reaction temperature gave the highest mean yield across replicate runs?

You've spent 30 minutes cleaning. Now watch what one join and one groupby buy you.

## Setup — install packages

Run this once per Colab session.

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Point these at YOUR cleaned CSVs. The three files below are the gold-standard
# safety net if your cleaning broke this notebook.
TIDY = 'sprint-dataset-tidy'  # adjust if you saved elsewhere
reactions = pd.read_csv(f'{TIDY}/reactions_log_tidy.csv')
catalysts = pd.read_csv(f'{TIDY}/catalyst_reference_tidy.csv')
solvents  = pd.read_csv(f'{TIDY}/solvent_reference_tidy.csv')

print(f'reactions: {len(reactions)} rows'); reactions.head()

In [ ]:
catalysts

In [ ]:
solvents

## Base analysis — yield by (catalyst × temperature)

Two operations: **join** the reactions to the catalyst reference on `catalyst_canonical`, then **groupby** (catalyst, temperature bin) and take the mean.

In [ ]:
df = reactions.merge(catalysts, on='catalyst_canonical', how='left')
df['temp_bin'] = (df['temperature_C'] // 20) * 20   # 20-C bins
answer = (df.groupby(['catalyst_canonical', 'temp_bin'])['yield_pct']
            .mean().unstack().round(1))
answer

In [ ]:
plt.figure(figsize=(9, 5))
sns.heatmap(answer, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': 'mean yield (%)'})
plt.title('Mean yield by catalyst × temperature bin')
plt.ylabel('catalyst'); plt.xlabel('temperature bin (deg C)')
plt.tight_layout()

**Callback.** Same research question you had 60 minutes ago at the sprint. How long did that take to answer with the messy file? How long does it take now?

---

## Bonus analyses (uncomment to run)

These require the *second* join — to `solvent_reference`. If your solvent-reference cleaning is broken, either fix it now or grab the gold-standard `solvent_reference_tidy.csv`.

In [ ]:
# BONUS 1 — join solvents in
# full = df.merge(solvents, on='solvent_canonical', how='left')
# print(f'after solvent join: {len(full)} rows (should match {len(df)})')

In [ ]:
# BONUS 2 — yield by solvent family
# (full.groupby('family')['yield_pct'].mean().sort_values(ascending=False)
#      .plot(kind='barh', title='mean yield by solvent family'))

In [ ]:
# BONUS 3 — yield vs solvent boiling point
# plt.figure(figsize=(9,5))
# sns.scatterplot(data=full, x='boiling_point_C', y='yield_pct',
#                 hue='catalyst_canonical', s=80)
# plt.title('yield vs. solvent boiling point'); plt.tight_layout()

In [ ]:
# BONUS 4 — catalyst x solvent grid
# grid = full.groupby(['catalyst_canonical','solvent_canonical'])['yield_pct'].mean().unstack()
# plt.figure(figsize=(9,5))
# sns.heatmap(grid, annot=True, fmt='.0f', cmap='YlGnBu')
# plt.title('mean yield: catalyst x solvent'); plt.tight_layout()